# 🏋️‍♂️ Edge-AI Gym Form Analyzer with Local VLMs

This notebook demonstrates how to build a resource-efficient, local video analysis pipeline to evaluate workout form using **OpenCV** and a local Vision-Language Model (**bonsai-2:4b** via **Ollama**).

### 💡 Motivation & Future Applications
By optimizing frame-extraction intervals and downscaling visual payloads, we can run complex visual reasoning pipelines on **resource-constrained edge devices** (such as smart security cameras, wearable tech, and drones). This unlocks low-latency, private, and cost-effective physical movement analysis at scale without relying on expensive cloud infrastructure.

In [ ]:
import requests

try:
    response = requests.get("http://localhost:11434/")
    if response.status_code == 200:
        print("Ollama background service is active and running! 🚀")
    else:
        print("Ollama responded, but with an unexpected status code.")
except requests.exceptions.ConnectionError:
    print("Ollama is NOT running. Please launch the Ollama desktop app or service.")

Ollama background service is active and running! 🚀


## 🛠️ Step 1: Video Processing & Inference Setup

To make this run efficiently on edge architectures:
1. We capture the video file and compute its exact frame-rate properties.
2. Instead of processing every frame, we sample key phases of movement (e.g., once every 2 seconds).
3. To stay well within the local context limit, we resize frames to **640x360** and serialize them to a base64 payload.
4. We then stream this structured payload to the lightweight **bonsai-2:4b** vision model.

In [ ]:
import cv2
import os
import base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

def analyze_gym_video(video_path, prompt):
    if not os.path.exists(video_path):
        print(f"Error: video file '{video_path}' not found, please check the path.")

    print(f"\n[+] opening video: {video_path}")
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 0
    print(f"[+] Video stats: {fps:.2f} FPS | Duration : {duration: .2f} seconds")


    # This will keep token size reasonable while preserving key movement phases
    sample_interval_sec = 2
    frame_interval = int(fps * sample_interval_sec) if fps > 0 else 30

    extracted_frames = []

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            #resize down to 320 x 180 for efficient edge simulation processing
            frame = cv2.resize(frame, (640,360))
            temp_filename = f"temp_frame_{len(extracted_frames)}.jpg"
            cv2.imwrite(temp_filename, frame)
            extracted_frames.append(temp_filename)
        frame_count += 1

    cap.release()

    # Hard cap to max 2 frames to stay under moondream:v2's 2048 token limit
    #if len(extracted_frames) > 2:
     #   extracted_frames = [extracted_frames[0], extracted_frames[-1]] # Keeps start and end frames

    print(f"[+] Extracted {len(extracted_frames)} keyframes from 15 second block")

    try:

        print("[+] Converting frames to base64 payload.....")

        def encode_image_to_base64(path):
            with open(path, "rb") as image_file:
                encoded_str = base64.b64encode(image_file.read()).decode("utf-8")
            return encoded_str

        #building the message payload
        message_content = [{"type":"text", "text":prompt}]
        for f in extracted_frames:
            encoded_str = encode_image_to_base64(f)
            message_content.append({
                "type":"image_url",
                "image_url": {"url" : f"data:image/jpeg;base64,{encoded_str}"}
            })


        print("[+] Intializing ChatOllama with bonsai-2:4b on GPU...")
        llm = ChatOllama(model="bonsai-2:4b", num_ctx=16384)

        print("[+] sending sequence to local VLM.....")
        response = llm.invoke([HumanMessage(content=message_content)])

        print("\n"+ "="*40)
        print("Gym form analysis result ")
        print("="*40)
        print(response.content)
        print("="*40 + "\n")

    except Exception as e:
        print(f"[-] An error occurred during inference: {e}")

    finally:
        #clean up temporay frames from disk
        for f in extracted_frames:
            if os.path.exists(f):
                os.remove(f)



## 📹 Step 2: Target Video Validation
Let's verify and display the raw workout video locally.

In [ ]:
from IPython.display import Video
Video("C:/Users/Shivachetan Ulavi/Desktop/workout_test/squats.mp4", width=640, embed=True)

## 🚀 Step 3: Run Form Analysis
Now we pass our video path and analysis prompt to our pipeline.

In [ ]:
if __name__ ==  "__main__":
    target_video = "C:/Users/Shivachetan Ulavi/Desktop/workout_test/squats.mp4"

    custom_prompt = (
        "Analyze this sequence of frames from a gym workout clip. "
        "Describe the exercise being performed, evaluate form stability, "
        "and suggest any potential improvements."
    )

    analyze_gym_video(target_video, custom_prompt)


[+] opening video: C:/Users/Shivachetan Ulavi/Desktop/workout_test/squats.mp4
[+] Video stats: 20.06 FPS | Duration :  14.56 seconds
[+] Extracted 8 keyframes from 15 second block
[+] Converting frames to base64 payload.....
[+] Intializing ChatOllama with bonsai-2:4b on GPU...
[+] sending sequence to local VLM.....

Gym form analysis result 
Based on the visual evidence in the provided sequence, here is an analysis of the workout clip:

### **Exercise Identification**
The athlete is performing a **Back Squat**.
*   **Evidence:** The use of a power rack with an overhead barbell loaded with plates (visible yellow and green plates) confirms this.
*   **Progression:** The sequence appears to show the athlete preparing for the lift, getting into a deep squat position, and then ascending. Specifically, frames 1 and 4 show the athlete in a deep squat stance, while frames 2 through 8 show the athlete standing with the barbell positioned on their upper back/shoulders.

### **Form Stability Ev

## 📝 Project Summary & Batch Processing Insights

In the next phase of this project (implemented in a standalone `.py` script), we scaled this pipeline to process **all 23 workout videos** asynchronously.

### ⚡ Scaling with Concurrent Execution
* **Mechanism:** We utilized Python's `ThreadPoolExecutor` to handle concurrent inferences on our local VLM (**bonsai-2:4b**).
* **Batch Configuration:** Inferences were processed in **batches of 2** to balance local GPU/CPU hardware compute limits and prevent out-of-memory (OOM) exceptions, ensuring high-throughput on an edge simulation.

### 📊 Model Performance & Observations
While lightweight VLMs are highly promising for edge-computing, we observed distinct performance patterns during bulk analysis:

* **🟢 Where it excelled (Standard/Linear Exercises):**
  The model demonstrated excellent accuracy, posture recognition, and structural form feedback on classic, highly linear lifts such as the **Deadlift** and **Pull-Ups**.
* **🔴 Where it struggled (Complex Movements / Edge Cases):**
  On complex or highly dynamic exercises like **Walking Lunges** and **Weighted Dips**, the model exhibited hallucinations (such as misidentifying exercise phases or recommending incorrect joint alignments).

*These findings suggest that while small 4B VLMs are extremely capable on-device engines, they can be further optimized for specific edge scenarios using targeted fine-tuning or prompt scaffolding.*